# Deep Agents - Context Isolation (Sub-agents Delegation)
This notebook implements specialized **Sub-agents** and a **Supervisor orchestrator** using **init_chat_model** (Gemini 2.5 Flash) with custom state reducers. Bypasses all Pydantic tool-state validation bugs.

In [1]:
# 1. Install required packages
#!pip install -qU langgraph langchain-google-genai pydantic typing_extensions python-dotenv

import os
from dotenv import load_dotenv

# 2. Load API key from .env file
print("🔄 Loading API key from .env file...")
load_dotenv()  # Auto-loads keys from your local .env file

if "GOOGLE_API_KEY" not in os.environ or not os.environ["GOOGLE_API_KEY"]:
    print("❌ ERROR: GOOGLE_API_KEY not found!")
    print("Please check if your .env file exists and contains: GOOGLE_API_KEY=your_key")
else:
    print("✅ API Key successfully loaded! Move to the next cell.")

🔄 Loading API key from .env file...
✅ API Key successfully loaded! Move to the next cell.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from typing import Annotated, Dict
from typing_extensions import TypedDict
from langchain_core.messages import ToolMessage, BaseMessage
from langchain_core.tools import tool, InjectedToolCallId
from langgraph.prebuilt import InjectedState, create_react_agent
from langgraph.types import Command
from langgraph.graph.message import add_messages

# 1. Reducer for Virtual File Tracking
def file_reducer(left: dict | None, right: dict | None) -> dict:
    if left is None:
        return right or {}
    if right is None:
        return left or {}
    return {**left, **right}

# 2. State definition with reducer (V1.0+ Compatible)
class DeepAgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    files: Annotated[Dict[str, str], file_reducer]
    remaining_steps: int  
    is_last_step: bool    

# 3. Basic Filesystem Tools for sub-agents
@tool
def ls(state: Annotated[dict, InjectedState] = None) -> list[str]:
    """List all files in the virtual filesystem."""
    if state is None:
        return []
    return list(state.get("files", {}).keys())

@tool
def read_file(file_path: str, state: Annotated[dict, InjectedState] = None) -> str:
    """Read content from a file in the virtual filesystem."""
    if state is None:
        return "Error: No state injected"
    files = state.get("files", {})
    if file_path not in files:
        return f"Error: File '{file_path}' not found"
    return files[file_path]

@tool
def write_file(
    file_path: str, 
    content: str, 
    tool_call_id: Annotated[str, InjectedToolCallId] = None
) -> Command:
    """Write or overwrite content to a file in the virtual filesystem."""
    return Command(
        update={
            "files": {file_path: content},
            "messages": [ToolMessage(f"Successfully saved content to {file_path}", tool_call_id=tool_call_id)],
        }
    )

# 4. Mock Web Search Tool for the researcher agent (No extra keys required)
@tool
def web_search(query: str) -> str:
    """Perform a web search query on a topic and return raw information."""
    print(f"🔍 [Search Tool executing query: '{query}']")
    return (
        f"Web Search results for: '{query}'\n"
        "1. AI Agents are moving towards multi-agent setups with strict context isolation.\n"
        "2. Frameworks like LangGraph use specialized sub-agents to divide complex tasks.\n"
        "3. Context offloading reduces token costs by writing research drafts to a virtual filesystem instead of conversation history."
    )

print("✅ Base components and State configured successfully.")

✅ Base components and State configured successfully.


In [3]:
# =====================================================================
# 🎯 STEP 1: CONFIGURE SPECIALIZED SUB-AGENTS (Using init_chat_model)
# =====================================================================

from langchain.chat_models import init_chat_model

# Initialize Gemini 2.5 Flash using LangChain's standardized initializer
llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)

# Subagent 1: Researcher (has web_search and write_file)
researcher_agent = create_react_agent(
    model=llm,
    tools=[web_search, write_file],
    state_schema=DeepAgentState,
    prompt="""You are a specialized Sub-Agent Researcher.
Your goal is to search the web using `web_search`, extract relevant details, and write a complete draft in the virtual file system using `write_file`.
Do not compile summaries—just perform the research and save it."""
)

# Subagent 2: Writer (has read_file and write_file)
writer_agent = create_react_agent(
    model=llm,
    tools=[read_file, write_file],
    state_schema=DeepAgentState,
    prompt="""You are a specialized Sub-Agent Writer/Editor.
Your goal is to read raw files from the filesystem using `read_file`, format and polish the contents, and write a polished 1-line report back using `write_file`."""
)

# Sub-agent Registry Mapping
subagents = {
    "researcher": researcher_agent,
    "writer": writer_agent
}

print("✅ Specialized Sub-agents successfully compiled via init_chat_model.")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ Specialized Sub-agents successfully compiled via init_chat_model.


C:\Users\ARYAN\AppData\Local\Temp\ipykernel_3500\60055287.py:11: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  researcher_agent = create_react_agent(
C:\Users\ARYAN\AppData\Local\Temp\ipykernel_3500\60055287.py:21: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  writer_agent = create_react_agent(


In [4]:
# =====================================================================
# 🎯 STEP 2: CREATE DYNAMIC TASK DELEGATION TOOL (With Pydantic Workaround)
# =====================================================================

description_prefix = (
    "Delegate a task to a specialized sub-agent with isolated context. "
    "Available agent types and descriptions:\n"
    "- `researcher`: Specialized in finding raw facts and writing draft files.\n"
    "- `writer`: Specialized in reading draft files and summarizing/polishing reports."
)

@tool(description=description_prefix)
def task(
    description: str,
    subagent_type: str,
    state: Annotated[dict, InjectedState] = None,
    tool_call_id: Annotated[str, InjectedToolCallId] = None
) -> Command:
    """Delegate a task to a specialized sub-agent with isolated context."""
    if subagent_type not in subagents:
        return Command(
            update={
                "messages": [ToolMessage(f"Error: Unknown agent type '{subagent_type}'", tool_call_id=tool_call_id)]
            }
        )
    
    print(f"\n👉 [Supervisor delegating task to sub-agent: '{subagent_type}']")
    sub_agent = subagents[subagent_type]
    
    # Create shallow copy of state with a fresh conversation context (isolated context)
    sub_state = dict(state) if state else {}
    sub_state["messages"] = [{"role": "user", "content": description}]
    sub_state["remaining_steps"] = 20
    sub_state["is_last_step"] = False
    
    # Invoke sub-agent
    result = sub_agent.invoke(sub_state)
    subagent_response = result["messages"][-1].content
    
    # Return result to the supervisor and merge modified files back
    return Command(
        update={
            "files": result.get("files", {}),
            "messages": [ToolMessage(subagent_response, tool_call_id=tool_call_id)],
        }
    )

print("✅ Dynamic task delegation tool compiled.")

✅ Dynamic task delegation tool compiled.


In [5]:
# =====================================================================
# 🎯 STEP 3: BUILD SUPERVISOR AGENT
# =====================================================================

supervisor_prompt = """You are a supervisor agent coordinating specialized sub-agents.
When given a multi-step request, you must divide the work and delegate to the appropriate sub-agents using the `task` tool.
Always delegate research tasks to `researcher` first to generate raw draft files, and then delegate formatting/summarization to `writer`.
Do not attempt to write the files or do the research yourself—delegate!"""

supervisor_agent = create_react_agent(
    model=llm,
    tools=[task],
    state_schema=DeepAgentState,
    prompt=supervisor_prompt
)

print("✅ Supervisor compiled! Ready for execution.")

✅ Supervisor compiled! Ready for execution.


C:\Users\ARYAN\AppData\Local\Temp\ipykernel_3500\838929756.py:10: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  supervisor_agent = create_react_agent(


In [7]:
# =====================================================================
# 🎯 STEP 4: MANAGER DEMO RUN (Delegation in Action)
# =====================================================================

# user_input = (
#     "Step 1: Have the researcher agent research 'AI Agent trends' and save the results to 'ai_trends.txt'. "
#     "Step 2: Have the writer agent read 'ai_trends.txt' and write a polished 1-line summary saved in 'report_summary.txt'."
# )
#test prompt(just to show 3 researcher agent get populated,no research will be done)
# user_input = (
#     "Step 1: Spawn 3 separate researcher tasks in parallel to research 3 different stocks: 'Tesla', 'Apple', and 'Nvidia', "
#     "and save the results to 'tesla.txt', 'apple.txt', and 'nvidia.txt' respectively. "
#     "Step 2: Once all 3 parallel research tasks are completed, have the writer agent read all 3 files and create a consolidated summary in 'market_report.txt'."
# )

print(f"🚀 Running Supervisor Agent for task:\n{user_input}\n")
print("=" * 60)

inputs = {
    "messages": [("user", user_input)],
    "files": {} 
}

final_event = None
for event in supervisor_agent.stream(inputs, stream_mode="values"):
    message = event["messages"][-1]
    message.pretty_print()
    final_event = event

print("\n" + "=" * 60)
print("📁 FINAL VIRTUAL FILESYSTEM STATE (Created by Sub-agents):")
print("=" * 60)

if final_event and "files" in final_event:
    for filename, content in final_event["files"].items():
        print(f"\n📄 File: {filename}")
        print("-" * 30)
        print(content)
        print("-" * 30)

🚀 Running Supervisor Agent for task:
Step 1: Spawn 3 separate researcher tasks in parallel to research 3 different stocks: 'Tesla', 'Apple', and 'Nvidia', and save the results to 'tesla.txt', 'apple.txt', and 'nvidia.txt' respectively. Step 2: Once all 3 parallel research tasks are completed, have the writer agent read all 3 files and create a consolidated summary in 'market_report.txt'.

================================ Human Message =================================

Step 1: Spawn 3 separate researcher tasks in parallel to research 3 different stocks: 'Tesla', 'Apple', and 'Nvidia', and save the results to 'tesla.txt', 'apple.txt', and 'nvidia.txt' respectively. Step 2: Once all 3 parallel research tasks are completed, have the writer agent read all 3 files and create a consolidated summary in 'market_report.txt'.
================================== Ai Message ==================================
Tool Calls:
  task (0df33eb1-6bc9-4e3d-8bb7-23bcd844f3b8)
 Call ID: 0df33eb1-6bc9-4e3d-8bb7